# Walkthrough: from portraits to grayscale distributions

This notebook runs the `colorism` pipeline step by step on a folder of `.jpg` portraits
(`data/images/` by default) and shows what each stage does to an image. Run it from the
repository root (`jupyter lab notebooks/walkthrough.ipynb`); see the README for ways to
fill the folder.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

import random
import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.offsetbox import AnnotationBbox, OffsetImage

import colorism

ROOT = os.path.dirname(colorism.__path__[0])
IMAGE_DIR = os.path.join(ROOT, 'data', 'images')
OUT_DIR = os.path.join(ROOT, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)
image_paths = colorism.list_images(IMAGE_DIR)
len(image_paths)

## 1. One image through the pipeline

Detection runs on a histogram-equalised grayscale copy; the statistics are computed on the
central 60% of the detected face box, converted to grayscale. The three panels are the input,
the colour crop, and the grayscale crop the numbers come from.

In [ ]:
face = np.nan
while not isinstance(face, np.ndarray):
    path = random.choice(image_paths)
    face = colorism.extract_face(path)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)); axes[0].set_title('input')
axes[1].imshow(cv2.cvtColor(face, cv2.COLOR_BGR2RGB)); axes[1].set_title('face crop')
axes[2].imshow(cv2.cvtColor(face, cv2.COLOR_BGR2GRAY), cmap='gray', vmin=0, vmax=255); axes[2].set_title('grayscale')
for ax in axes:
    ax.axis('off')
dict(zip(colorism.STAT_COLUMNS, colorism.get_greyscale_stats(face)))

## 2. The whole folder

Images under 120 px, images with no detected face, and crops whose mean or mode falls
outside 30-230 (nearly always a false detection) come back as NaN and are dropped.

In [ ]:
stats_df = colorism.batch_process(IMAGE_DIR)
valid = stats_df.dropna(subset=colorism.STAT_COLUMNS).reset_index(drop=True)
print(f'{len(stats_df)} images, {len(valid)} usable face crops')
colorism.summarise(valid).round(1)

In [ ]:
colorism.plot_distributions(valid, os.path.join(OUT_DIR, 'distributions.png'))
plt.imshow(plt.imread(os.path.join(OUT_DIR, 'distributions.png'))); plt.axis('off');

## 3. Faces on the intensity scale

Each crop is placed at its modal grayscale value (y) with a little horizontal jitter, so you
can see what the numbers correspond to and spot crops that are hair, background or glare.

In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(0, 1, len(valid))
fig, ax = plt.subplots(figsize=(6, 10))
ax.scatter(x, valid['mode'], s=0)
for xi, yi, crop in zip(x, valid['mode'], valid['face_image']):
    thumb = cv2.cvtColor(cv2.resize(crop, (48, 48)), cv2.COLOR_BGR2RGB)
    ax.add_artist(AnnotationBbox(OffsetImage(thumb, zoom=0.6), (xi, yi), frameon=False))
ax.set_xticks([]); ax.set_ylim(0, 255); ax.set_ylabel('modal grayscale value of crop');

## 4. Comparing groups

If you have a label for each image (for example FairFace's `race` column), merge it on the
file name and compare. `python -m colorism ... --labels ... --group ...` does the same from
the command line.

In [ ]:
LABELS = None  # e.g. os.path.join(ROOT, 'data', 'fairface', 'fairface_label_val.csv')
GROUP = 'race'
if LABELS:
    import pandas as pd
    labels = pd.read_csv(LABELS)
    labels['file'] = labels['file'].map(os.path.basename)
    grouped = valid.drop(columns='face_image').merge(labels, on='file')
    colorism.plot_group_comparison(grouped, GROUP, os.path.join(OUT_DIR, f'by_{GROUP}.png'))
    display(colorism.group_summary(grouped, GROUP).round(1))